In [0]:
storage_key = dbutils.secrets.get(
    scope="azure-storage",
    key="storage-key"
)

spark.conf.set(
    "fs.azure.account.key.azurestorageteju.dfs.core.windows.net",
    storage_key
)

In [0]:

df = spark.read.csv(
    "abfss://destination@azurestorageteju.dfs.core.windows.net/bronze/orders.csv",
    header=True,
    inferSchema=True
)

df.show()

In [0]:
df.count()

In [0]:
df = df.dropDuplicates()

In [0]:
df = df.na.drop()
df.show()

In [0]:
df.write \
.mode("overwrite") \
.format("delta") \
.save("abfss://destination@azurestorageteju.dfs.core.windows.net/silver/orders.csv")

In [0]:
orders = spark.read \
.option("header","true") \
.csv("abfss://destination@azurestorageteju.dfs.core.windows.net/bronze/orders.csv")

customers = spark.read \
.option("header","true") \
.csv("abfss://destination@azurestorageteju.dfs.core.windows.net/bronze/customers.csv")

products = spark.read \
.option("header","true") \
.csv("abfss://destination@azurestorageteju.dfs.core.windows.net/bronze/products.csv")

In [0]:
customers = customers.dropDuplicates().dropna()

In [0]:
products = products.dropDuplicates()

In [0]:
orders = orders.filter("status='Completed'")

In [0]:
silver = orders.join(
    customers,
    "customer_id",
    "inner"
).join(
    products,
    "product_id",
    "inner"
)

In [0]:
silver.show()

In [0]:
silver.write \
.mode("overwrite") \
.format("delta") \
.save("abfss://destination@azurestorageteju.dfs.core.windows.net/silver/sales.csv")

In [0]:
silver = spark.read.format("delta").load(
"abfss://destination@azurestorageteju.dfs.core.windows.net/silver/sales.csv")

In [0]:
from pyspark.sql.functions import sum,col

gold = silver.groupBy("product_name").agg(
    sum(
        col("quantity").cast("int") *
        col("price").cast("double")
    ).alias("Total_Sales")
)

gold.show()

In [0]:
from pyspark.sql.functions import count

gold = silver.groupBy("customer_id").agg(
    count("order_id").alias("Total_Orders")
)

gold.show()

In [0]:
from pyspark.sql.functions import count

gold = silver.groupBy("status").agg(
    count("*").alias("Total_Orders")
)

gold.show()

In [0]:
gold.write \
.mode("overwrite") \
.format("delta") \
.save("abfss://destination@azurestorageteju.dfs.core.windows.net/gold/orders_summary.csv")

In [0]:
gold_read = spark.read \
.format("delta") \
.load("abfss://destination@azurestorageteju.dfs.core.windows.net/gold/orders_summary.csv")

gold_read.show()